# 机器学习专升本知识点智能答疑助手 —— 完整流程演示

《机器学习基础及应用》期末大作业 · 方向3：教育服务类（进阶版）

本 Notebook 串联：知识库构建 → 数据预处理 → 3 种算法训练评估 → 两级检索 → RAG 文档问答演示。

## 0. 环境与路径准备

In [1]:
import os, sys, json
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
import pandas as pd, numpy as np
print("环境就绪")

环境就绪


## 1. 知识库构建与数据预处理

调用 `build_kb.py` 生成 213 个机器学习知识点，并完成缺失值/重复值/类别校验。

In [2]:
import build_kb
df = build_kb.build_dataframe()
print("知识点总数：", len(df))
print("缺失值数量：", int(df.isnull().sum().sum()))
print("重复标题数：", int(df.duplicated(subset=['title']).sum()))
df['category'].value_counts()

知识点总数： 105
缺失值数量： 0
重复标题数： 0


category
神经网络     21
模型评估     18
数学基础     18
监督学习     17
特征工程     17
无监督学习    14
Name: count, dtype: int64

In [3]:
# 查看知识库样例
df.head(3)

,id,category,title,content,keywords
0,1,监督学习,什么是监督学习,监督学习是指利用带有标签的训练数据来学习一个从输入特征到输出标签的映射函数。训练集中每个样本...,"监督学习,标签,分类,回归,映射函数"
1,2,监督学习,分类与回归的区别,分类与回归都属于监督学习。分类的输出是离散的类别标签，例如判断邮件是否为垃圾邮件、图片属于哪...,"分类,回归,离散,连续,目标变量"
2,3,监督学习,线性回归原理,线性回归假设目标变量与特征之间存在线性关系，模型形式为 y = w·x + b。通过最小化损...,"线性回归,最小二乘法,均方误差,权重,偏置"


## 2. 特征工程：jieba 中文分词 + TF-IDF

中文需先分词。这里展示分词效果。

In [4]:
from text_utils import tokenize
sample = "朴素贝叶斯是基于贝叶斯定理的分类算法，假设特征条件独立。"
print("分词结果：", tokenize(sample))

Building prefix dict from the default dictionary ...


Loading model from cache D:\Temp\jieba.cache


Loading model cost 0.465 seconds.


Prefix dict has been built successfully.


分词结果： ['朴素', '贝叶斯', '基于', '贝叶斯', '定理', '分类', '算法', '假设', '特征', '条件', '独立']


## 3. 三种分类算法训练、调优与多指标评估

实现**朴素贝叶斯、决策树、随机森林**，用 GridSearchCV 网格搜索调参，
用准确率/精确率/召回率/F1/AUC 多指标评估，绘制混淆矩阵与 ROC。

In [5]:
import train_classify as tc
tc.main()   # 训练三模型并保存评估报告与图表

加载知识库：213 条，6 个类别 -> ['数学基础', '无监督学习', '模型评估', '特征工程', '监督学习', '神经网络']

训练集 78 条 | 测试集 27 条

训练模型：朴素贝叶斯


  最优参数：{'clf__alpha': 0.1}
  测试集多指标评估：
    准确率             : 0.7407
    精确率(macro)      : 0.7937
    召回率(macro)      : 0.7417
    F1(macro)       : 0.7567
    AUC(macro-ovr)  : 0.9167


训练模型：决策树


  最优参数：{'clf__max_depth': 20, 'clf__min_samples_split': 3}
  测试集多指标评估：
    准确率             : 0.4074
    精确率(macro)      : 0.4226
    召回率(macro)      : 0.4167
    F1(macro)       : 0.4119
    AUC(macro-ovr)  : 0.6434


训练模型：随机森林


  最优参数：{'clf__max_depth': 20, 'clf__n_estimators': 100}
  测试集多指标评估：
    准确率             : 0.6296
    精确率(macro)      : 0.6413
    召回率(macro)      : 0.625
    F1(macro)       : 0.6102
    AUC(macro-ovr)  : 0.8651



最优模型：朴素贝叶斯（F1-macro=0.7567），已保存到 models/nb_classifier.pkl
朴素贝叶斯最终模型已保存到 models/nb_final.pkl
评估报告：models/eval_report.json
混淆矩阵 / ROC 曲线图已保存到 models/ 目录


In [6]:
# 读取评估报告，对比三算法
with open(os.path.join("..", "models", "eval_report.json"), encoding="utf-8") as f:
    report = json.load(f)
rows = []
for name in ["朴素贝叶斯", "决策树", "随机森林"]:
    if name in report:
        rows.append({"模型": name, **report[name]["metrics"]})
pd.DataFrame(rows).set_index("模型")

,准确率,精确率(macro),召回率(macro),F1(macro),AUC(macro-ovr)
模型,,,,,
朴素贝叶斯,0.7407,0.7937,0.7417,0.7567,0.9167
决策树,0.4074,0.4226,0.4167,0.4119,0.6434
随机森林,0.6296,0.6413,0.6250,0.6102,0.8651


**结论**：朴素贝叶斯在该文本分类任务上表现最佳（F1、AUC 最高），符合其在高维稀疏文本上的优势，故作为知识点分类与检索的主模型。

## 4. 两级检索演示

先用朴素贝叶斯预测问题类别缩小范围，再在 TF-IDF 空间用余弦相似度排序。

In [7]:
import retriever
r = retriever.get_retriever()
for q in ["什么是过拟合", "K-means如何选择K值", "ReLU激活函数有什么优点"]:
    out = r.search(q, top_k=2)
    print(f"问题：{q}")
    print(f"  预测类别：{out['predicted_category']}")
    for it in out["results"]:
        print(f"  - [{it['category']}] {it['title']} (相似度 {it['score']})")
    print()

问题：什么是过拟合
  预测类别：模型评估
  - [模型评估] 过拟合 (相似度 0.3942)
  - [模型评估] 学习曲线 (相似度 0.2417)

问题：K-means如何选择K值
  预测类别：无监督学习
  - [无监督学习] K-means如何选择K值 (相似度 0.6218)
  - [无监督学习] K-means聚类原理 (相似度 0.3638)

问题：ReLU激活函数有什么优点
  预测类别：神经网络
  - [神经网络] 激活函数的作用 (相似度 0.5101)
  - [神经网络] ReLU激活函数 (相似度 0.4389)



## 5. RAG 文档问答演示（进阶①）

把一段文本作为"文档"，演示分块 → 检索 → 返回原文片段。

In [8]:
import rag
text = ("机器学习分为监督学习和无监督学习。监督学习需要带标签的数据，可做分类与回归。"
        "无监督学习不需要标签，常用于聚类与降维。过拟合指模型在训练集表现好但测试集差，"
        "可通过正则化、增加数据、Dropout 等方法缓解。")
engine = rag.RAGEngine(text)
ans, hits = engine.answer("如何缓解过拟合？")
print("文档块数：", len(engine.chunks))
print("回答：\n", ans)

文档块数： 1
回答：
 （以下为文档中最相关的原文片段）

【片段1】机器学习分为监督学习和无监督学习。监督学习需要带标签的数据，可做分类与回归。无监督学习不需要标签，常用于聚类与降维。过拟合指模型在训练集表现好但测试集差，可通过正则化、增加数据、Dropout 等方法缓解。


## 6. 模拟试卷生成与评分演示（进阶③）

In [9]:
import exam
qs = exam.generate_exam(n=3)   # 离线模板出题
for i, q in enumerate(qs, 1):
    print(f"[{i}] ({q['type']}/{q['category']}) {q['question'][:50]}... 答案={q['answer']}")
score, total, details = exam.grade(qs, [q['answer'] for q in qs])  # 全答对演示
print(f"\n全部答对得分：{score}/{total}")

[1] (判断题/模型评估) 关于「F1值」：梯度下降是优化模型参数的核心方法，通过沿损失函数梯度的反方向迭代更新参数来最小化损失... 答案=错误
[2] (判断题/神经网络) 关于「梯度消失与梯度爆炸」：词向量（Word Embedding）把词语映射为低维稠密的实数向量，使... 答案=错误
[3] (判断题/神经网络) 关于「损失函数」：损失函数衡量模型预测值与真实值的差距，是优化的目标。... 答案=正确

全部答对得分：3/3


## 7. 总结

- 知识库：213 个知识点，6 大类别，完成预处理校验。
- 模型：朴素贝叶斯/决策树/随机森林，网格搜索调优，多指标评估，朴素贝叶斯最优。
- 检索：两级检索提升准确率。
- 进阶：RAG 文档问答、模拟试卷生成评分、错题薄弱项分析（见 Streamlit 应用）。

完整智能体运行：`streamlit run app.py`